In [ ]:
%load_ext autoreload
%autoreload 2
    
from dataclasses import dataclass
from google.cloud import storage
from scipy.ndimage import gaussian_filter as blur
from tqdm.notebook import tqdm
from PIL import Image

import base64
import io
import json
import numpy as np
import os
import pandas as pd
import voxeloo
from voxeloo import mapping, geometry

In [ ]:
def visualize_voxels(voxels, wireframe=False):
    if voxels.dtype == bool:
        voxels = np.array([0, 0xffffffff])[voxels.astype(int)]
    
    mesh = voxeloo.voxels.voxels_to_mesh(voxels)
    
    # Convert the mesh into the trimesh format.
    tm = trimesh.Trimesh(
        vertices=mesh.vertices[:, 0:3],
        faces=mesh.triangles,
    )
    
    # Add the vertex colors.
    tm.visual.face_colors = mesh.vertices[mesh.triangles[:, 0], 3:6]
    
    scene = pyrender.Scene(ambient_light=[0.2, 0.2, 0.2, 1.0])
    scene.add(
        pyrender.Mesh.from_trimesh(
            tm,
            smooth=False,
            wireframe=wireframe,
        )
    )
    pyrender.Viewer(
        scene,
        use_raymond_lighting=True,
    )

In [ ]:
@dataclass
class TerrainVoxel:
    name: str
    index: int
    color: int

In [ ]:
def to_flora_id(terrain_id):
    return (1 << 24) | (terrain_id & 0xffffff)

# These IDs match the terrain IDs in: src/galois/js/assets/blocks.ts
terrain = [
    TerrainVoxel("none", 0, 0x00000000),
    TerrainVoxel("basalt", 8, 0x323143ff),
    TerrainVoxel("bedrock", 6, 0x4d3f53ff),
    TerrainVoxel("birch_log", 12, 0xb9c8d2ff),
    TerrainVoxel("clay", 17, 0x755a56ff),
    TerrainVoxel("coal_ore", 19, 0x2f2d34ff),
    TerrainVoxel("cobblestone", 5, 0x706e76ff),
    TerrainVoxel("diamond_ore", 23, 0x36d9f1ff),
    TerrainVoxel("dirt", 2, 0x9b633cff),
    TerrainVoxel("gold_ore", 11, 0xded02cff),
    TerrainVoxel("granite", 13, 0xb2bcc7ff),
    TerrainVoxel("grass", 1, 0x30a144ff),
    TerrainVoxel("gravel", 36, 0xad8c64ff),
    TerrainVoxel("hay", 35, 0xcdb158ff),
    TerrainVoxel("limestone", 9, 0x8b7b70ff),
    TerrainVoxel("moss", 40, 0x247447ff),
    TerrainVoxel("neptunium_ore", 30, 0x394f57ff),
    TerrainVoxel("oak_log", 3, 0x774530ff),
    TerrainVoxel("pumpkin", 10, 0xc67a2bff),
    TerrainVoxel("quartzite", 7, 0x527d79ff),
    TerrainVoxel("rubber_log", 14, 0x603526ff),
    TerrainVoxel("silver_ore", 25, 0x7f9cbaff),
    TerrainVoxel("stone", 4, 0x828690ff),
    TerrainVoxel("snow", 37, 0xe5f6faff),

    # Crafted blocks
    TerrainVoxel("basalt_brick", 15, 0x323143ff),
    TerrainVoxel("basalt_carved", 41, 0x323143ff),
    TerrainVoxel("basalt_polished", 42, 0x323143ff),
    TerrainVoxel("basalt_shingles", 43, 0x323143ff),
    TerrainVoxel("birch_lumber", 16, 0xc29b44ff),
    TerrainVoxel("clay_brick", 18, 0x883b49ff),
    TerrainVoxel("clay_carved", 44, 0x883b49ff),
    TerrainVoxel("clay_polished", 45, 0x883b49ff),
    TerrainVoxel("clay_shingles", 46, 0x883b49ff),
    TerrainVoxel("cobblestone_brick", 20, 0x706e76ff),
    TerrainVoxel("cobblestone_carved", 47, 0x706e76ff),
    TerrainVoxel("cobblestone_polished", 48, 0x706e76ff),
    TerrainVoxel("cobblestone_shingles", 49, 0x706e76ff),
    TerrainVoxel("cotton_fabric", 38, 0xe9e8e9ff),
    TerrainVoxel("diamond", 21, 0xa6f6ffff),
    TerrainVoxel("gold", 24, 0xf7ca00ff),
    TerrainVoxel("granite_brick", 26, 0xb2bcc7ff),
    TerrainVoxel("granite_carved", 50, 0xb2bcc7ff),
    TerrainVoxel("granite_polished", 51, 0xb2bcc7ff),
    TerrainVoxel("granite_shingles", 52, 0xb2bcc7ff),
    TerrainVoxel("limestone_brick", 27, 0x8b7b70ff),
    TerrainVoxel("limestone_carved", 53, 0x8b7b70ff),
    TerrainVoxel("limestone_polished", 54, 0x8b7b70ff),
    TerrainVoxel("limestone_shingles", 55, 0x8b7b70ff),
    TerrainVoxel("mushroom_leather", 39, 0x7c5a45ff),
    TerrainVoxel("neptunium", 28, 0x224a3eff),
    TerrainVoxel("oak_lumber", 31, 0x8b572aff),
    TerrainVoxel("quartzite_brick", 32, 0x527d79ff),
    TerrainVoxel("quartzite_carved", 56, 0x527d79ff),
    TerrainVoxel("quartzite_polished", 57, 0x527d79ff),
    TerrainVoxel("quartzite_shingles", 58, 0x527d79ff),
    TerrainVoxel("rubber_lumber", 34, 0x764330ff),
    TerrainVoxel("silver", 33, 0xccd6dbff),
    TerrainVoxel("stone_brick", 29, 0x828690ff),
    TerrainVoxel("stone_carved", 59, 0x828690ff),
    TerrainVoxel("stone_polished", 60, 0x828690ff),
    TerrainVoxel("stone_shingles", 61, 0x828690ff),
    TerrainVoxel("template", 69, 0x990099ff),
    TerrainVoxel("wood_crate", 22, 0x8b572aff),

    # Flora
    TerrainVoxel("oak_leaf", to_flora_id(1), 0x125d37ff),
    TerrainVoxel("birch_leaf", to_flora_id(2), 0x4e6a1dff),
    TerrainVoxel("rubber_leaf", to_flora_id(3), 0x20575aff),
    TerrainVoxel("switch_grass", to_flora_id(4), 0x4bbb47ff),
    TerrainVoxel("azalea_flower", to_flora_id(5), 0xc86c8dff),
    TerrainVoxel("bell_flower", to_flora_id(6), 0x59a5d5ff),
    TerrainVoxel("dandelion_flower", to_flora_id(7), 0xd0c227ff),
    TerrainVoxel("daylily_flower", to_flora_id(8), 0xcd8216ff),
    TerrainVoxel("lilac_flower", to_flora_id(9), 0xa767f2ff),
    TerrainVoxel("rose_flower", to_flora_id(10), 0xcc374cff),
    TerrainVoxel("cotton_flower", to_flora_id(11), 0xe9e8e9ff),
    TerrainVoxel("hemp_bush", to_flora_id(12), 0x2b6230ff),
    TerrainVoxel("red_mushroom", to_flora_id(13), 0xb02f3aff),
]

terrain_index = {tv.name: tv.index for tv in terrain}

terrain_color = np.zeros(max(tv.index for tv in terrain) + 1, dtype=np.uint32)
for tv in terrain:
    terrain_color[tv.index] = tv.color

In [ ]:
def hex_to_rgb(array):
    return np.stack(
        [
            (array >> 24) & 0xff,
            (array >> 16) & 0xff,
            (array >> 8) & 0xff,
        ],
        axis=-1,
    ).astype(np.uint8)

### Load the world data

In [ ]:
client = storage.Client()
bucket = client.get_bucket("biomes-experimental")

In [ ]:
%%time 

blob = bucket.get_blob("biomes_alpha_large_2022_06_03.parquet")

In [ ]:
%%time

table_data = blob.download_as_string()

In [ ]:
%%time 

df = pd.read_parquet(io.BytesIO(table_data))

In [ ]:
%%time 

terrain_blobs = []
for row in tqdm(range(len(df))):
    try:
        delta = df["delta"][row]
        try:
            state = json.loads(delta["state"])[1]
            if 34 not in state[1:] or 35 not in state[1:]:
                state = json.loads(df["state"][row])
        except:
            state = json.loads(df["state"][row])
        if 34 in state[1:] and 35 in state[1:]:
            terrain_blobs.append(
                (
                    state[0],
                    state[state.index(33) + 1],
                    state[state.index(34) + 1][0],
                    state[state.index(35) + 1][1],
                    state[state.index(60) + 1][0],
                )
            )
    except:
        continue

In [ ]:
%%time 

# Range of the world over which the map is generated.
aabb = [
    [-512, -224, -512],
    [ 512,  288,  512],
]

# Terrain IDs omitted from the map.
omitted_voxels = [
    "switch_grass",
    "azalea_flower",
    "bell_flower",
    "dandelion_flower",
    "daylily_flower",
    "lilac_flower",
    "rose_flower",
    "cotton_flower",
    "hemp_bush",
    "red_mushroom",  
]

# Build the map by loading each shard within the AABB.
builder = mapping.SurfaceBuilder({
    terrain_index[name] for name in omitted_voxels
})

for key, box, seed, diff, shapes in tqdm(terrain_blobs):
    block = voxeloo.biomes.VolumeBlock_U32()
    block.loads(seed)
    
    if diff:
        delta = voxeloo.biomes.SparseBlock_U32()
        delta.loads(diff)
        block.assign(delta)

    if np.all(np.array(box[0]) >= aabb[0]) and np.all(np.array(box[1]) <= aabb[1]):
        builder.load_shard(box[0], block)
        
surface = builder.build()

### Render the height map

In [ ]:
heights = np.array(surface.heights, copy=False)
Image.fromarray(np.clip(heights, 0, 256).astype(np.uint8))

### Render the colors map

In [ ]:
colors = terrain_color[np.array(surface.ids, copy=False)]
Image.fromarray(hex_to_rgb(colors))

### Render the map with a simple shading model.

In [ ]:
dz, dx = np.gradient(heights)

normals = np.zeros(shape=(*heights.shape, 3), dtype=float)
normals[:, :, 0] = dx
normals[:, :, 1] = 1.0
normals[:, :, 2] = dz

normals /= np.sqrt(normals[:, :, 0]**2 + normals[:, :, 1]**2, normals[:, :, 2]**2)[:, :, np.newaxis]

shading = np.sum(normals * np.array([0.57735, 0.57735, 0.57735])[np.newaxis, np.newaxis, :], axis=-1)
shading = np.clip(0.4 + 0.8 * shading, 0.3, 1)

final_map = hex_to_rgb(colors).astype(float) / 255.0
final_map[:, :, 0] *= shading
final_map[:, :, 1] *= shading
final_map[:, :, 2] *= shading

final_map = np.clip(final_map, 0, 1)

Image.fromarray((255 * final_map).astype(np.uint8))

### Load textures into a texture map

In [ ]:
def texture_from_blob(blob):
    data = base64.b64decode(blob)
    img = Image.open(io.BytesIO(data))
    return mapping.Array2_Vec3_U8(np.array(img)[..., :3])

def texture_from_color(color, shape=(16, 16)):
    colors = color * np.ones(shape=shape, dtype=np.uint32)
    return mapping.Array2_Vec3_U8(hex_to_rgb(colors))

In [ ]:
root = "../../../data"

builder = mapping.TextureMapBuilder()

# Add all block types.
for name, index in terrain_index.items():
    config_path = f"{root}/blocks/{name}.json"
    if os.path.exists(config_path):
        with open(config_path) as f:
            config = json.load(f)
            for sample in config["black"]:
                builder.add(index, mapping.Tile.Black, texture_from_blob(sample["color"]["y_pos"]))
            for sample in config["white"]:
                builder.add(index, mapping.Tile.White, texture_from_blob(sample["color"]["y_pos"]))
            
# Add all flora types.
for name in ["oak_leaf", "birch_leaf", "rubber_leaf"]:
    index = terrain_index[name]
    texture = texture_from_color(terrain_color[index])
    builder.add(index, mapping.Tile.Black, texture)
    builder.add(index, mapping.Tile.White, texture)
    
# Add water.
texture = texture_from_color(0x0000ffff)
builder.add(1337, mapping.Tile.Black, texture)
builder.add(1337, mapping.Tile.White, texture)
            
textures = builder.build()

### Render an upsampled version of the region of the map.

In [ ]:
def upsample(src, window, dpi):
    ret = np.zeros(
        shape=(
            dpi * (window[1][0] - window[0][0]),
            dpi * (window[1][1] - window[0][1]),
        ),
        dtype=src.dtype,
    )
    for y in range(dpi):
        for x in range(dpi):
            lo = window[0][0], window[0][1]
            hi = window[1][0], window[1][1]
            ret[y::dpi, x::dpi] = src[lo[0]:hi[0], lo[1]:hi[1]]
    return ret

In [ ]:
%%time 

DPI = 4
WINDOW = [
    [0, 0],
    [1024, 1024],
] 

up_heights = upsample(heights, WINDOW, DPI)
display(Image.fromarray(np.clip(up_heights, 0, 256).astype(np.uint8)))

up_colors = upsample(colors, WINDOW, DPI)
display(Image.fromarray(hex_to_rgb(up_colors)))

### Render a color map

In [ ]:
%%time

mat = mapping.render_textures(
    WINDOW,
    DPI,
    mapping.to_material_array(surface),
    textures,
)
Image.fromarray(np.array(mat, copy=False))

### Render an ambient occlusion map

In [ ]:
%%time

ao = mapping.render_light(
    WINDOW,
    DPI,
    mapping.to_ambient_occlusion_array(surface),
)
Image.fromarray(np.array(ao, copy=False))

### Render combined AO and colors

In [ ]:
%%time

out = np.array(mat) / 255.0
light = np.array(ao, copy=False) / 255.0          

out *= light

Image.fromarray(np.clip(255 * out, 0, 255).astype(np.uint8))

### Render a shadow map

In [ ]:
%%time

shadow = mapping.render_light(
    WINDOW,
    DPI,
    mapping.to_shadow_array(surface),
)
Image.fromarray(np.array(shadow, copy=False))

### Render combined AO, shadows, and colors

In [ ]:
%%time

out = np.array(mat) / 255.0 
light = np.multiply(
    0.5 * np.array(ao, copy=False) / 255.0 + 0.5,
    0.4 * np.array(shadow, copy=False) / 255.0 + 0.6
)

out[:, :] *= light

Image.fromarray(np.clip(255 * out, 0, 255).astype(np.uint8))

### Render it all directly

In [ ]:
%%time

out = mapping.render_surface(
    WINDOW,
    DPI,
    surface,
    textures,
    0.5,
    0.4,
    1,
)

Image.fromarray(np.array(out, copy=False))

### Try some quantization

In [ ]:
%%time

out = mapping.render_surface(
    WINDOW,
    DPI,
    surface,
    textures,
    0.5,
    0.4,
    4,
)

Image.fromarray(np.array(out, copy=False))